# Data scaling and optimization-budget sensitivity

A three-epoch curve changes both unique data and update count. The fixed-update grid holds optimizer updates constant. It does not claim equal FLOPs or equal audio exposure. Subsets are nested prefixes of one eligible training order.

In [ ]:
# Resolve shared code when launched from code/ or the repository root.
import sys
from pathlib import Path

_code_candidates = [Path.cwd(), Path.cwd() / "code"]
CODE = next(
    (
        p.resolve()
        for p in _code_candidates
        if (p / "neyshekar_experiments" / "protocol.py").is_file()
    ),
    None,
)
if CODE is None:
    raise RuntimeError("Launch this notebook from the repository root or its code/ directory.")
if str(CODE) not in sys.path:
    sys.path.insert(0, str(CODE))
from neyshekar_experiments.protocol import ROOT
# End notebook bootstrap

import pandas as pd
from IPython.display import display
from neyshekar_experiments.manifests import prepare
from neyshekar_experiments.training import experiment_grid, plan, execute
from neyshekar_experiments.reporting import family_scores, corrected_results, metric_figure

# Imports and reports never start training; paths use the shared repository root.

RUN_TRAINING = False
DATA_SEED = 42  # All manifests are frozen with this seed; optimization seeds are separate.

# Create/verify manifests from source data when this notebook is run.
manifest_summary = prepare()

## 1. Fresh WER and CER

Plots appear only after runs complete. Show each optimization budget separately.

In [ ]:
fresh = family_scores("scaling")
display(fresh)
if not fresh.empty:
    for _, group in fresh.groupby(["budget", "max_steps"], dropna=False):
        metric_figure(group)

## 2. Three-epoch data curve

Both grids select the same frozen subsets. The complete eligible pool is recorded in the manifest.

In [ ]:
epoch_runs = experiment_grid("scaling", seeds=(42,))
display(plan(epoch_runs))

## 3. Fixed-update sensitivity

The common update budget is three epochs of the official Common Voice training split (1,425 updates at batch 64). Small subsets are revisited more often. Actual update count and elapsed training time are recorded.

In [ ]:
update_runs = experiment_grid("scaling_updates", seeds=(42,))
display(plan(update_runs))
if RUN_TRAINING:
    execute(epoch_runs + update_runs)

In [ ]:
display(corrected_results())